In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight
import re
import json

In [4]:
print("\n=== Stage 1: Data Loading and Preparation ===")

# Load the raw ticket dataset
df_raw = pd.read_csv("/content/customer_support_tickets.csv")

# Align column names with the schema used throughout the pipeline
column_mapping = {
    "ticket_priority": "priority_level",
    "time_to_resolution": "resolution_time_hours",
    "ticket_type": "issue_category",
    "ticket_subject": "ticket_subject"
}

df_raw.rename(columns=column_mapping, inplace=True)

# Standardize column naming for easier downstream processing
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

# Ensure resolution time is numeric; invalid entries become NaN
df_raw["resolution_time_hours"] = pd.to_numeric(
    df_raw["resolution_time_hours"],
    errors="coerce"
)

# Normalize priority labels to a consistent format
priority_lookup = {
    "low": "Low",
    "medium": "Medium",
    "high": "High",
    "critical": "Critical"
}

df_raw["priority_level"] = (
    df_raw["priority_level"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(priority_lookup)
)

# Keep only records that contain all fields required by the model
required_fields = [
    "ticket_description",
    "ticket_subject",
    "priority_level",
    "ticket_channel",
    "resolution_time_hours",
    "issue_category"
]

df = (
    df_raw
    .dropna(subset=required_fields)
    .reset_index(drop=True)
)

# Preserve the original priority distribution in both splits
df_train, df_val = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["priority_level"]
)

print(f"Training samples   : {len(df_train)}")
print(f"Validation samples : {len(df_val)}")

print("\n=== Stage 2: Signal Fusion and Pseudo-Label Generation ===")

# Category-wise median resolution time acts as a reference baseline
train_medians = (
    df_train
    .groupby("issue_category")["resolution_time_hours"]
    .median()
    .to_dict()
)


=== Stage 1: Data Loading and Preparation ===
Training samples   : 16000
Validation samples : 4000

=== Stage 2: Signal Fusion and Pseudo-Label Generation ===


In [5]:
def infer_text_severity(ticket_text):
    """
    Estimate severity directly from the language used in the ticket.
    This acts as a lightweight rule-based signal independent of metadata.
    """
    ticket_text = str(ticket_text).lower()

    critical_pattern = r"\b(outage|breach|lawsuit|security|fraud|wiped|gone|down|fatal)\b"
    high_pattern = r"\b(urgent|asap|escalate|manager|unacceptable|broken|crash|immediately)\b"
    medium_pattern = r"\b(delay|waiting|cancel|refund|error|issue|problem|help|stuck)\b"

    if re.search(critical_pattern, ticket_text):
        return "Critical"

    if re.search(high_pattern, ticket_text):
        return "High"

    if re.search(medium_pattern, ticket_text):
        return "Medium"

    return "Low"


def infer_resolution_severity(resolution_hours, category):
    """
    Infer severity based on how quickly a ticket was resolved relative
    to the historical median for its category.
    """
    category_median = train_medians.get(category, resolution_hours)

    ratio = (
        resolution_hours / category_median
        if category_median > 0
        else 1
    )

    # Extremely urgent issues are often resolved much faster than normal.
    if ratio < 0.35:
        return "Critical"

    if ratio < 0.75:
        return "High"

    if ratio > 1.50:
        return "Low"

    return "Medium"

In [6]:
severity_to_score = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Critical": 3
}

score_to_severity = {
    value: key
    for key, value in severity_to_score.items()
}

# Work on copies to avoid modifying the original split objects
df_train = df_train.copy()
df_val = df_val.copy()

# Independent severity estimates from text and resolution behavior
df_train["text_signal"] = (
    df_train["ticket_description"]
    .apply(infer_text_severity)
)

df_train["resolution_signal"] = df_train.apply(
    lambda row: infer_resolution_severity(
        row["resolution_time_hours"],
        row["issue_category"]
    ),
    axis=1
)

# Combine both signals into a single pseudo-label.
# Text carries more weight because it contains the actual issue context.
df_train["pseudo_severity_score"] = np.round(
    (
        df_train["text_signal"].map(severity_to_score) * 0.80
    ) +
    (
        df_train["resolution_signal"].map(severity_to_score) * 0.20
    )
).astype(int)

df_train["pseudo_severity"] = (
    df_train["pseudo_severity_score"]
    .map(score_to_severity)
)

# Measure how much the two independent signals agree with each other
signal_agreement = cohen_kappa_score(
    df_train["text_signal"],
    df_train["resolution_signal"]
)

print(f"Signal Agreement (Cohen's Kappa): {signal_agreement:.4f}")

Signal Agreement (Cohen's Kappa): 0.0022


In [7]:
def generate_mismatch_label(assigned_priority, inferred_priority):
    """
    Returns:
        1 -> priorities do not match
        0 -> priorities are consistent
    """
    assigned_priority = str(assigned_priority).strip().capitalize()
    inferred_priority = str(inferred_priority).strip().capitalize()

    return int(assigned_priority != inferred_priority)


# Create the target label used for binary classification
df_train["binary_mismatch_label"] = df_train.apply(
    lambda row: generate_mismatch_label(
        row["priority_level"],
        row["pseudo_severity"]
    ),
    axis=1
)

print("\n=== Stage 3: Feature Construction ===")


def create_model_input(row):
    """
    Build a single text sequence that combines structured metadata
    with the ticket content before feeding it to DistilBERT.
    """
    return (
        f"Priority={row['priority_level']} | "
        f"Channel={row['ticket_channel']} | "
        f"Subject={row['ticket_subject']} | "
        f"Description={row['ticket_description']}"
    )


=== Stage 3: Feature Construction ===


In [8]:
# Convert training records into model-ready text inputs
train_texts = (
    df_train
    .apply(create_model_input, axis=1)
    .tolist()
)

train_labels = df_train["binary_mismatch_label"].tolist()


# Recreate the same pseudo-labeling pipeline on the validation split
df_val["text_signal"] = (
    df_val["ticket_description"]
    .apply(infer_text_severity)
)

df_val["resolution_signal"] = df_val.apply(
    lambda row: infer_resolution_severity(
        row["resolution_time_hours"],
        row["issue_category"]
    ),
    axis=1
)

df_val["pseudo_severity_score"] = np.round(
    (
        df_val["text_signal"].map(severity_to_score) * 0.80
    ) +
    (
        df_val["resolution_signal"].map(severity_to_score) * 0.20
    )
).astype(int)

df_val["pseudo_severity"] = (
    df_val["pseudo_severity_score"]
    .map(score_to_severity)
)

df_val["binary_mismatch_label"] = df_val.apply(
    lambda row: generate_mismatch_label(
        row["priority_level"],
        row["pseudo_severity"]
    ),
    axis=1
)

val_texts = (
    df_val
    .apply(create_model_input, axis=1)
    .tolist()
)

val_labels = df_val["binary_mismatch_label"].tolist()

print("\n=== Stage 4: DistilBERT Initialization ===")

# Tokenize the combined ticket representations
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=256
)


=== Stage 4: DistilBERT Initialization ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
class SupportTicketDataset(Dataset):
    """
    Lightweight dataset wrapper that serves tokenized inputs
    and labels in a format expected by Hugging Face Trainer.
    """

    def __init__(self, tokenized_inputs, targets):
        self.tokenized_inputs = tokenized_inputs
        self.targets = targets

    def __getitem__(self, index):
        batch_item = {
            feature_name: torch.tensor(feature_values[index])
            for feature_name, feature_values in self.tokenized_inputs.items()
        }

        batch_item["labels"] = torch.tensor(
            self.targets[index],
            dtype=torch.long
        )

        return batch_item

    def __len__(self):
        return len(self.targets)


# Binary classifier: consistent vs mismatch
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


# Compute class weights to reduce bias toward the majority class
unique_classes = np.unique(train_labels)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=train_labels
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)


class WeightedLossTrainer(Trainer):
    """
    Custom trainer that applies weighted cross-entropy during training.
    This helps when the mismatch labels are not evenly distributed.
    """

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        loss_function = nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(outputs.logits.device)
        )

        loss = loss_function(
            outputs.logits.view(-1, 2),
            labels.view(-1)
        )

        if return_outputs:
            return loss, outputs

        return loss

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [12]:
from evaluate import load

# Macro-F1 is used because both classes matter equally
f1_evaluator = load("f1")


def compute_metrics(eval_predictions):
    raw_predictions, true_labels = eval_predictions

    predicted_labels = np.argmax(
        raw_predictions,
        axis=1
    )

    return f1_evaluator.compute(
        predictions=predicted_labels,
        references=true_labels,
        average="macro"
    )


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    report_to="none"
)


trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=SupportTicketDataset(
        train_encodings,
        train_labels
    ),
    eval_dataset=SupportTicketDataset(
        val_encodings,
        val_labels
    ),
    compute_metrics=compute_metrics
)


print("\n=== Stage 5: Model Training ===")

# Fine-tune DistilBERT on the mismatch detection task
trainer.train()


print("\n=== Stage 6: Validation and Evaluation ===")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== Stage 5: Model Training ===


Epoch,Training Loss,Validation Loss,F1
1,0.351235,0.347362,0.863938
2,0.354890,0.349811,0.863938
3,0.337593,0.342391,0.863938
4,0.337785,0.342021,0.863476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Stage 6: Validation and Evaluation ===


In [13]:
import torch.nn.functional as F

# Generate predictions on the validation split
validation_dataset = SupportTicketDataset(
    val_encodings,
    val_labels
)

prediction_output = trainer.predict(
    validation_dataset
)

prediction_probs = F.softmax(
    torch.tensor(prediction_output.predictions),
    dim=-1
).numpy()


# Search for the threshold that maximizes Macro-F1
candidate_thresholds = np.arange(0.30, 0.70, 0.01)

best_threshold = 0.50
best_macro_f1 = 0.0

for threshold in candidate_thresholds:

    candidate_predictions = (
        prediction_probs[:, 1] >= threshold
    ).astype(int)

    current_f1 = f1_score(
        val_labels,
        candidate_predictions,
        average="macro"
    )

    if current_f1 > best_macro_f1:
        best_macro_f1 = current_f1
        best_threshold = threshold


print(
    f"\nBest classification threshold: "
    f"{best_threshold:.2f}"
)

final_predictions = (
    prediction_probs[:, 1] >= best_threshold
).astype(int)


print(
    classification_report(
        val_labels,
        final_predictions,
        target_names=[
            "Consistent (0)",
            "Mismatch (1)"
        ],
        zero_division=0
    )
)


accuracy = accuracy_score(
    val_labels,
    final_predictions
)

macro_f1_score = f1_score(
    val_labels,
    final_predictions,
    average="macro"
)

consistent_recall = recall_score(
    val_labels,
    final_predictions,
    pos_label=0
)

mismatch_recall = recall_score(
    val_labels,
    final_predictions,
    pos_label=1
)

print("\n" + "=" * 55)
print(f"Accuracy            : {accuracy:.4f}")
print(f"Macro F1            : {macro_f1_score:.4f}")
print(f"Recall (Consistent) : {consistent_recall:.4f}")
print(f"Recall (Mismatch)   : {mismatch_recall:.4f}")
print("=" * 55)


print("\n=== Stage 7: Audit Report Generation ===")

# Confidence corresponds to the highest class probability
confidence_scores = np.max(
    prediction_probs,
    axis=1
)

audit_reports = []

for row_index, predicted_label in enumerate(final_predictions):

    if predicted_label != 1:
        continue

    confidence_value = float(
        confidence_scores[row_index]
    )

    if confidence_value >= 0.85:
        confidence_level = "High"
    elif confidence_value >= 0.65:
        confidence_level = "Medium"
    else:
        confidence_level = "Low"

    ticket_record = df_val.iloc[row_index]

    audit_reports.append(
        {
            "ticket_id": str(
                ticket_record.get(
                    "ticket_id",
                    f"VAL-{row_index}"
                )
            ),
            "assigned_priority": ticket_record["priority_level"],
            "audit_result": "Potential Priority Mismatch",
            "confidence_level": confidence_level,
            "confidence_score": round(
                confidence_value,
                4
            ),
            "supporting_evidence": [
                {
                    "signal": "ticket_channel",
                    "value": ticket_record["ticket_channel"]
                }
            ],
            "analysis": (
                f"Assigned priority "
                f"({ticket_record['priority_level']}) "
                f"does not align with the model's inferred severity."
            )
        }
    )


print(
    json.dumps(
        audit_reports[:2],
        indent=4
    )
)

# Persist model artifacts for inference/deployment
trainer.save_model("./model")
tokenizer.save_pretrained("./model")


Best classification threshold: 0.30
                precision    recall  f1-score   support

Consistent (0)       0.83      0.84      0.83      1539
  Mismatch (1)       0.90      0.89      0.89      2461

      accuracy                           0.87      4000
     macro avg       0.86      0.87      0.86      4000
  weighted avg       0.87      0.87      0.87      4000


Accuracy            : 0.8708
Macro F1            : 0.8639
Recall (Consistent) : 0.8408
Recall (Mismatch)   : 0.8895

=== Stage 7: Audit Report Generation ===
[
    {
        "ticket_id": "TKT-110456",
        "assigned_priority": "Medium",
        "audit_result": "Potential Priority Mismatch",
        "confidence_level": "Medium",
        "confidence_score": 0.7593,
        "supporting_evidence": [
            {
                "signal": "ticket_channel",
                "value": "Email"
            }
        ],
        "analysis": "Assigned priority (Medium) does not align with the model's inferred severity."
    }

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model/tokenizer_config.json', './model/tokenizer.json')

In [14]:
import os

print(os.listdir("./model"))

['tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'training_args.bin']


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

tokenizer.save_pretrained("./model")

('./model/tokenizer_config.json', './model/tokenizer.json')

In [16]:
import os

print(os.listdir("./model"))

['tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'training_args.bin']


In [17]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

tokenizer = AutoTokenizer.from_pretrained("./model")

model = AutoModelForSequenceClassification.from_pretrained(
    "./model"
)

print("Loaded successfully")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded successfully


In [18]:
trainer.save_model("./model")
tokenizer.save_pretrained("./model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model/tokenizer_config.json', './model/tokenizer.json')

In [19]:
!zip -r model.zip model

  adding: model/ (stored 0%)
  adding: model/tokenizer.json (deflated 71%)
  adding: model/model.safetensors (deflated 8%)
  adding: model/config.json (deflated 48%)
  adding: model/tokenizer_config.json (deflated 43%)
  adding: model/training_args.bin (deflated 54%)


In [20]:
from google.colab import files

files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from transformers import AutoTokenizer

base_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

base_tokenizer.save_pretrained("./model")

('./model/tokenizer_config.json', './model/tokenizer.json')

In [24]:
import os
print(os.listdir("./model"))

['tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'training_args.bin']


In [25]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./model")

print(type(tokenizer))
print("Tokenizer loaded successfully")

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
Tokenizer loaded successfully


In [26]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_DIR = "./model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

sample_text = (
    "Priority=Low | "
    "Channel=Email | "
    "Subject=Server outage | "
    "Description=Production systems are down."
)

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

outputs = model(**inputs)

print(outputs.logits)
print("Deployment test passed")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.2213, -0.9926]], grad_fn=<AddmmBackward0>)
Deployment test passed
